In [0]:
from pyspark.sql.functions import *

# Read Bronze
df = spark.table("retailer.bronze.customers_raw")

# Birthday Check using try_to_date for tolerance
df = df.withColumn(
    "birthday",
    coalesce(
        try_to_date(col("birthday"), "M/d/yyyy"),
        try_to_date(col("birthday"), "dd-MM-yyyy") 
    )
)

# 3. Remove null keys
df = df.dropna(subset=["customer_key"])

# 4. Remove duplicates
df = df.dropDuplicates(["customer_key"])

# 5. Standardize gender
df = df.withColumn(
    "gender",
    when(col("gender").isin("M", "Male"), "Male")
    .when(col("gender").isin("F", "Female"), "Female")
    .otherwise("Unknown")
)

# 6. Clean location
df = df.withColumn("country", initcap(col("country"))) \
       .withColumn("state", initcap(col("state"))) \
       .withColumn("city", initcap(col("city")))

# 7. Zip code fix
df = df.withColumn("zip_code", col("zip_code").cast("string"))

# 8. Create age
df = df.withColumn(
    "age",
    floor(months_between(current_date(), col("birthday")) / 12)
)

df_final = df.select(
    "customer_key",
    "name",
    "gender",
    "age",
    "city",
    "state",
    "country",
    "zip_code",
    "continent",
    "birthday",
)

# 13. Write to Silver
df_final.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailer.silver.customers")

print("✅ Silver Table Created")

In [0]:
df = spark.table("retailer.silver.customers")
df.show()